# TimesFM - Foundation Time Series Model (Bonus)
imports, set up wandb.

In [ ]:
from pathlib import Path
import sys
import time

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import wandb
import timesfm

from src.preprocessing import load_raw, build_features, wmae
from src.config import WANDB_ENTITY, WANDB_PROJECT
from src.wandb_utils import safe_wandb_init, save_and_log_pipeline_artifact

np.random.seed(42)
print("W&B entity:", WANDB_ENTITY)
print("W&B project:", WANDB_PROJECT)

REPO_ID = "google/timesfm-2.5-200m-pytorch"

## 1. Load & Reshape Data

In [2]:
train, test, features, stores = load_raw()
train_full = build_features(train, features, stores)

long_df = train_full[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].rename(
    columns={"Date": "ds", "Weekly_Sales": "y"}
)
long_df["unique_id"] = long_df["Store"].astype(str) + "_" + long_df["Dept"].astype(str)
long_df = long_df[["unique_id", "ds", "y", "IsHoliday"]].sort_values(["unique_id", "ds"])

print(long_df.shape)
long_df.head()

(421570, 4)


,unique_id,ds,y,IsHoliday
87524,10_1,2010-02-05,40212.84,False
87525,10_1,2010-02-12,67699.32,True
87526,10_1,2010-02-19,49748.33,False
87527,10_1,2010-02-26,33601.22,False
87528,10_1,2010-03-05,36572.44,False


## 2. Filter Series

In [3]:
H = 39
INPUT_SIZE = 52
MIN_LEN = INPUT_SIZE + H

series_len = long_df.groupby("unique_id").size()
keep_ids = series_len[series_len >= MIN_LEN].index

print(f"series total: {series_len.shape[0]}")
print(f"series kept (len >= {MIN_LEN}): {len(keep_ids)}")
print(f"series dropped (too short): {series_len.shape[0] - len(keep_ids)}")

long_df = long_df[long_df["unique_id"].isin(keep_ids)].reset_index(drop=True)
print("rows after filtering:", long_df.shape[0])

with safe_wandb_init(
    run_name="TimesFM_Preprocessing",
    group="TimesFM",
    job_type="preprocessing",
    config={"horizon_h": H, "input_size": INPUT_SIZE, "min_series_length": MIN_LEN},
) as run:
    wandb.log({
        "series_total": series_len.shape[0],
        "series_kept": len(keep_ids),
        "series_dropped": series_len.shape[0] - len(keep_ids),
        "rows_kept": long_df.shape[0],
    })

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


series total: 3331
series kept (len >= 91): 2900
series dropped (too short): 431
rows after filtering: 410069


wandb: Currently logged in as: abeku20 (abeku20-free-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run 9r0o4ztk


wandb: Tracking run with wandb version 0.28.0


...

wandb: updating run metadata; uploading summary


wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json


wandb: 
wandb: Run history:
wandb:      rows_kept ▁
wandb: series_dropped ▁
wandb:    series_kept ▁
wandb:   series_total ▁
wandb: 
wandb: Run summary:
wandb:      rows_kept 410069
wandb: series_dropped 431
wandb:    series_kept 2900
wandb:   series_total 3331
wandb: 


wandb:  View run TimesFM_Preprocessing at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/9r0o4ztk
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_172716-9r0o4ztk\logs


## 3. Zero-Shot Holdout Evaluation

In [4]:
contexts, holds = [], []
for uid, g in long_df.groupby("unique_id"):
    g = g.sort_values("ds")
    ctx, hold = g.iloc[-(H + INPUT_SIZE):-H], g.iloc[-H:]
    if len(ctx) == INPUT_SIZE and len(hold) == H:
        contexts.append(ctx["y"].values.astype(np.float32))
        holds.append(hold)

print(f"series with a full context+holdout window: {len(contexts)}")

t0 = time.time()
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(REPO_ID)
load_seconds = time.time() - t0
print(f"model load took {load_seconds:.1f} sec")

model.compile(timesfm.ForecastConfig(
    max_context=INPUT_SIZE,
    max_horizon=H,
    normalize_inputs=True,
    use_continuous_quantile_head=False,
    force_flip_invariance=True,
    infer_is_positive=True,
    fix_quantile_crossing=True,
))

t0 = time.time()
point, quantiles = model.forecast(horizon=H, inputs=contexts)
forecast_seconds = time.time() - t0
print(f"zero-shot forecast for {len(contexts)} series took {forecast_seconds:.1f} sec")

series with a full context+holdout window: 2900


model load took 3.2 sec


zero-shot forecast for 2900 series took 350.1 sec


In [5]:
hold_df = pd.concat(holds).reset_index(drop=True)
hold_df["TimesFM"] = point.reshape(-1)

val_wmae = wmae(hold_df["y"], hold_df["TimesFM"], hold_df["IsHoliday"])
val_mae = (hold_df["y"] - hold_df["TimesFM"]).abs().mean()

print(f"Validation WMAE: {val_wmae:.2f}")
print(f"Validation MAE:  {val_mae:.2f}")

with safe_wandb_init(
    run_name="TimesFM_ZeroShot_CV",
    group="TimesFM",
    job_type="cross_validation",
    config={"repo_id": REPO_ID, "horizon_h": H, "input_size": INPUT_SIZE, "fine_tuned": False},
) as run:
    wandb.log({
        "model_load_seconds": load_seconds,
        "forecast_seconds": forecast_seconds,
        "val_wmae": val_wmae,
        "val_mae": val_mae,
    })

Validation WMAE: 2507.66
Validation MAE:  2429.92


wandb: setting up run ub84gcqq


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_173315-ub84gcqq
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run TimesFM_ZeroShot_CV


...

wandb: uploading summary


wandb: uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:   forecast_seconds ▁
wandb: model_load_seconds ▁
wandb:            val_mae ▁
wandb:           val_wmae ▁
wandb: 
wandb: Run summary:
wandb:   forecast_seconds 350.12807
wandb: model_load_seconds 3.2257
wandb:            val_mae 2429.9167
wandb:           val_wmae 2507.66445
wandb: 


wandb:  View run TimesFM_ZeroShot_CV at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/ub84gcqq
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_173315-ub84gcqq\logs


## 4. Pipeline Export

In [6]:
from src.dl_models import TimesFMForecastPipeline

with safe_wandb_init(
    run_name="TimesFM_Final",
    group="TimesFM",
    job_type="final_training",
    config={"repo_id": REPO_ID, "horizon_h": H, "input_size": INPUT_SIZE, "fine_tuned": False},
) as run:
    wandb.log({"val_wmae": val_wmae})

    save_and_log_pipeline_artifact(
        run,
        "timesfm-pipeline",
        files={"train.csv": "data/raw/train.csv"},
        metadata={
            "architecture": "TimesFM",
            "metric": "WMAE",
            "val_wmae": val_wmae,
            "repo_id": REPO_ID,
            "horizon_h": H,
            "input_size": INPUT_SIZE,
            "fine_tuned": False,
            "uses_raw_test": True,
            "notes": "Zero-shot: no local weights, just a reference to the frozen pretrained "
                     "checkpoint (downloaded from Hugging Face Hub at load time) plus train.csv "
                     "for reconstructing each series' recent history. Load via "
                     "src.dl_models.TimesFMForecastPipeline.load(train_csv_path).",
        },
        aliases=["latest", "candidate"],
    )
    print("Logged pipeline to W&B run:", run.id)

wandb: setting up run f6cty11l


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_173331-f6cty11l
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run TimesFM_Final


wandb:  View project at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast


...

wandb: uploading artifact timesfm-pipeline; uploading requirements.txt


wandb: uploading artifact timesfm-pipeline


wandb: 
wandb: Run history:
wandb: val_wmae ▁
wandb: 
wandb: Run summary:
wandb: val_wmae 2507.66445
wandb: 


wandb:  View run TimesFM_Final at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/f6cty11l
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_173331-f6cty11l\logs
